In [10]:
import re
import pandas as pd
import json

# Step 1: Load predictions and targets from CSV file
df = pd.read_csv("inference_results.csv")

# Universal float pattern: matches 12, 4.2, 0.8 etc.
FLOAT_PATTERN = r"(\d+(?:\.\d+)?)"

# Helper function to extract information from the text
def extract_fields(text):
    state_match = re.search(r"(?P<state>cat|thermal|coherent|fock|random|number) state", text, re.IGNORECASE)

    alpha_match   = re.search(r"(?:α|alpha)\s*(?:≈|=)\s*(\d+(?:\.\d+)?)", text, re.IGNORECASE)
    density_match = re.search(r"(?:with\s*)?density(?:\s*value)?\s*(?:≈|=)\s*(\d+(?:\.\d+)?)", text, re.IGNORECASE)
    photon_match  = re.search(r"(?:average\s*)?photon[s]?\s*(?:≈|=)\s*(\d+(?:\.\d+)?)", text, re.IGNORECASE)

    qubits_match  = re.search(r"qubits\s*=\s*(\d+)", text)
    linear_match  = re.search(r"linear space.*?(?:from\s*)?-?(\d+)\s*to\s*-?(\d+)", text, re.IGNORECASE)

    param = None
    if alpha_match:
        param = float(alpha_match.group(1))
    elif density_match:
        param = float(density_match.group(1))
    elif photon_match:
        param = float(photon_match.group(1))

    return {
        "state": state_match.group("state").lower() if state_match else None,
        "parameter": param,
        "qubits": int(qubits_match.group(1)) if qubits_match else None,
        "linear_space": abs(int(linear_match.group(2))) if linear_match else None
    }

# Process all records from DataFrame
output = []
for idx, row in df.iterrows():
    output.append({
        "index": idx,
        "generated_text": extract_fields(row["generated"]),
        "ground_truth": extract_fields(row["ground_truth"])
    })

# Save to JSON file
with open("wigner_states_base_cleaned.json", "w") as f:
    json.dump(output, f, indent=2)


In [11]:
import json

# Sample input data
# Step 1: Load predictions and targets from JSON file
with open("wigner_states_base_cleaned.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Fields to compare
fields = ["state", "parameter", "qubits", "linear_space"]

# Initialize counters
field_totals = {field: 0 for field in fields}
field_totals["all_correct"] = 0
total = len(data)

# Count matches
for item in data:
    gen = item["generated_text"]
    gt = item["ground_truth"]
    all_correct = True

    for field in fields:
        if gen.get(field) == gt.get(field):
            field_totals[field] += 1
        else:
            all_correct = False

    if all_correct:
        field_totals["all_correct"] += 1

# Compute accuracy
field_accuracies = {field: correct / total for field, correct in field_totals.items()}

# Print results
print("Accuracy by field:")
for field, acc in field_accuracies.items():
    print(f"  {field}: {acc:.2%}")


Accuracy by field:
  state: 85.17%
  parameter: 24.03%
  qubits: 7.64%
  linear_space: 58.84%
  all_correct: 3.04%
